# Проверка модели

Этот notebook проверяет расчет AHP-весов, `Score` и `Rank`.

In [1]:
from importlib import import_module
from pathlib import Path
import sys

# Определяем корень проекта.
project_root = Path.cwd()
if not (project_root / "data" / "trade.xlsx").exists():
    project_root = project_root.parent

# Добавляем корень проекта в пути импорта.
sys.path.insert(0, str(project_root))

loader = import_module("src.1_data_loader.loader")
preprocessing = import_module("src.2_preprocessing.preprocessing")
indicators = import_module("src.3_indicators.indicators")
normalization = import_module("src.4_normalization.normalization")
model = import_module("src.5_model.model")

In [2]:
# Выполняем полный расчет до модели.
calculation_year = 2025
clipping_mode = "1-99"

raw_data = loader.load_trade_data(project_root / "data" / "trade.xlsx")
prepared_data = preprocessing.preprocess_trade_data(raw_data)
yearly_trade = preprocessing.make_yearly_trade_table(prepared_data)
country_import = preprocessing.make_country_import_table(prepared_data)
indicator_values = indicators.calculate_indicators(yearly_trade, country_import, calculation_year)
normalized_values = normalization.normalize_indicators(indicator_values, clipping_mode)

ranking, ahp_info = model.calculate_priority_ranking(normalized_values)

In [3]:
# Проверяем AHP-веса и согласованность.
weights = ahp_info["weights"]

assert set(weights.keys()) == {"C1", "C2", "C3", "C4"}
assert abs(sum(weights.values()) - 1) < 1e-9
assert ahp_info["consistency_ratio"] <= 0.10

ahp_info

{'weights': {'C1': 0.42311507936507936,
  'C2': 0.12251984126984128,
  'C3': 0.22718253968253968,
  'C4': 0.22718253968253968},
 'lambda_max': 4.010365356719721,
 'consistency_index': 0.0034551189065735364,
 'consistency_ratio': 0.0038390210073039293,
 'is_consistent': np.True_}

In [4]:
# Проверяем итоговую таблицу ранжирования.
assert "Score" in ranking.columns
assert "Rank" in ranking.columns
assert ranking["Score"].between(0, 1).all()
assert ranking["Rank"].min() == 1
assert ranking["Score"].is_monotonic_decreasing

ranking.head(10)

,TNVED,Year,Import,Export,C1,C2,C3,C4,normalized_C1,normalized_C2,normalized_C3,normalized_C4,Score,Rank
0,851762,2025,5.045269e+08,39581140.73,0.036268,-0.219754,0.927255,0.744869,1.000000,0.554947,0.923445,0.693740,0.858503,1
1,847130,2025,6.652730e+08,32341918.68,0.047823,-0.627314,0.953639,0.726230,1.000000,0.507959,0.951212,0.671365,0.853971,2
2,852872,2025,4.898981e+08,28018886.65,0.035216,-0.115642,0.945901,0.698877,0.987323,0.566950,0.943068,0.638530,0.846525,3
3,850440,2025,3.947554e+08,20611264.47,0.028377,0.032828,0.950378,0.913684,0.795575,0.584067,0.947780,0.896386,0.827142,4
4,851713,2025,1.642379e+09,24592845.87,0.118061,-0.354816,0.985247,0.517682,1.000000,0.539375,0.984476,0.421024,0.808504,5
5,847150,2025,4.134439e+08,5763428.13,0.029720,-0.847561,0.986252,0.536008,0.833239,0.482567,0.985533,0.443022,0.736223,6
6,847330,2025,3.362190e+08,2099259.06,0.024169,0.014902,0.993795,0.695460,0.677602,0.582000,0.993472,0.634429,0.727841,7
7,901890,2025,5.469254e+08,10861420.28,0.039315,0.158033,0.980528,0.169886,1.000000,0.598502,0.979509,0.003529,0.719772,8
8,850980,2025,2.543640e+08,1186490.97,0.018285,-0.176126,0.995357,0.876933,0.512634,0.559977,0.995116,0.852270,0.705205,9
9,850811,2025,2.131480e+08,7292440.44,0.015322,-0.024811,0.966919,0.826433,0.429569,0.577422,0.965187,0.791650,0.651625,10
